# 1. Import et nettoyage 

In [87]:
import hashlib, urllib.request
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

URL = "https://docs.ambroise.cloud/velib/velib-2026-10-08.csv.gz"
FILE = "velib-2026-10-08.csv.gz"
MD5 = "02ab1e3fdac10579e4ce93d1540a5dc7"

# 1. Download to disk, then verify the checksum
path, _ = urllib.request.urlretrieve(URL, FILE)
print("Download successful:", path)

print("------------------------------------------------------")

# 2. Verify MD5
with open(path, "rb") as f:
    calculated_md5 = hashlib.md5(f.read()).hexdigest()

print("Calculated MD5:", calculated_md5)
print("Expected MD5:  ", MD5)

assert calculated_md5 == MD5, f"MD5 mismatch: {calculated_md5}"
print("MD5 OK!")

print("------------------------------------------------------")

# 3. Read, then convert the time columns
raw = pd.read_csv(path)
print("\n Dataset shape:", raw.shape)
print("   Number of rows:", raw.shape[0])
print("   Number of columns:", raw.shape[1])

print("------------------------------------------------------")
print("\n Column names:")
print(raw.columns.tolist())


Download successful: velib-2026-10-08.csv.gz
------------------------------------------------------
Calculated MD5: 02ab1e3fdac10579e4ce93d1540a5dc7
Expected MD5:   02ab1e3fdac10579e4ce93d1540a5dc7
MD5 OK!
------------------------------------------------------

 Dataset shape: (329680, 16)
   Number of rows: 329680
   Number of columns: 16
------------------------------------------------------

 Column names:
['collected_at_utc', 'source_updated_at', 'station_code', 'name', 'commune', 'insee_code', 'latitude', 'longitude', 'capacity', 'bikes_available', 'mechanical_bikes', 'electric_bikes', 'docks_available', 'is_installed', 'is_renting', 'is_returning']


In [86]:
# 4. Convert time columns
for c in ["collected_at_utc", "source_updated_at"]:
    raw[c] = pd.to_datetime(raw[c], utc=True)

df = raw.copy()   # `raw` stays untouched; all cleaning happens on df
print(df.head())
print(df.dtypes[["collected_at_utc", "source_updated_at"]])

           collected_at_utc         source_updated_at  station_code                             name commune  insee_code   latitude  longitude  capacity  bikes_available  mechanical_bikes  \
0 2026-08-25 11:57:26+00:00 2026-08-25 11:35:29+00:00         10017         Gare de l'Est - Fidélité   Paris       75056  48.874575   2.356796      15.0                8                 1   
1 2026-08-25 11:57:26+00:00 2026-08-25 11:34:30+00:00         10042  Enghien - Faubourg Poissonnière   Paris       75056  48.872420   2.348395      33.0               30                22   
2 2026-08-25 11:57:26+00:00 2026-08-25 11:32:01+00:00         10163         Gare du Nord - Demarquay   Paris       75056  48.881154   2.358411      30.0                2                 0   
3 2026-08-25 11:57:26+00:00 2026-08-25 11:32:46+00:00         10202       Bichat - Grange aux Belles   Paris       75056  48.873688   2.365015      30.0                4                 3   
4 2026-08-25 11:57:26+00:00 2026-08-25 11:33:

# 2  Data's quality check 

In [69]:
# Les valeurs dupplicants
print("\n Number of duplicate rows:", df.duplicated().sum())


 Number of duplicate rows: 0


In [70]:
# 8. Check missing values
print("\nMissing values per column:")
print(df.isna().sum())


Missing values per column:
collected_at_utc        0
source_updated_at       0
station_code            0
name                    0
commune                 0
insee_code              0
latitude                0
longitude               0
capacity             1920
bikes_available         0
mechanical_bikes        0
electric_bikes          0
docks_available         0
is_installed            0
is_renting              0
is_returning            0
dtype: int64


In [71]:
missing_capacity = df[df["capacity"].isna()]

print(missing_capacity.head())

               collected_at_utc         source_updated_at  station_code                               name          commune  insee_code   latitude  longitude  capacity  bikes_available  \
64370 2026-09-02 22:30:01+00:00 2026-09-02 21:59:19+00:00         31104          Mairie de Rosny-sous-Bois  Rosny-sous-Bois       93064  48.871257   2.486581       NaN               29   
64371 2026-09-02 22:30:01+00:00 2026-09-02 21:58:54+00:00         32001        Proudhon - Président Wilson      Saint-Denis       93066  48.906443   2.358491       NaN                9   
64372 2026-09-02 22:30:01+00:00 2026-09-02 21:56:16+00:00          3201       Payenne - Square George Caïn            Paris       75056  48.857833   2.362264       NaN               13   
64373 2026-09-02 22:30:01+00:00 2026-09-02 21:56:30+00:00         32020             Jules Guesdes - Carnot      Saint-Denis       93066  48.938282   2.350899       NaN               14   
64374 2026-09-02 22:30:01+00:00 2026-09-02 21:50:11+00:00   

In [73]:
print(missing_capacity["name"].value_counts())

name
Mairie de Rosny-sous-Bois                24
Proudhon - Président Wilson              24
Payenne - Square George Caïn             24
Jules Guesdes - Carnot                   24
Square du Théâtre du Garde-Chasse        24
                                         ..
Labouret - Saint-Denis                   24
Richard Wallace - Place du 8 Mai 1945    24
Saint-Gilles - Turenne                   24
Archives - Pastourelle                   24
Lagny - Auguste Blanqui                  24
Name: count, Length: 80, dtype: int64


In [88]:
m = df[df["capacity"].isna()]
print("distinct timestamps:", m["collected_at_utc"].nunique())
print(sorted(m["collected_at_utc"].unique()))

distinct timestamps: 26
[Timestamp('2026-09-02 22:30:01+0000', tz='UTC'), Timestamp('2026-09-02 22:45:02+0000', tz='UTC'), Timestamp('2026-09-02 23:00:02+0000', tz='UTC'), Timestamp('2026-09-02 23:15:01+0000', tz='UTC'), Timestamp('2026-09-02 23:30:02+0000', tz='UTC'), Timestamp('2026-09-02 23:45:02+0000', tz='UTC'), Timestamp('2026-09-03 00:00:01+0000', tz='UTC'), Timestamp('2026-09-03 00:15:02+0000', tz='UTC'), Timestamp('2026-09-03 00:45:01+0000', tz='UTC'), Timestamp('2026-09-25 22:30:01+0000', tz='UTC'), Timestamp('2026-09-25 22:45:02+0000', tz='UTC'), Timestamp('2026-09-25 23:15:02+0000', tz='UTC'), Timestamp('2026-09-25 23:30:02+0000', tz='UTC'), Timestamp('2026-09-25 23:45:01+0000', tz='UTC'), Timestamp('2026-09-26 00:00:02+0000', tz='UTC'), Timestamp('2026-09-26 00:15:01+0000', tz='UTC'), Timestamp('2026-09-26 00:30:02+0000', tz='UTC'), Timestamp('2026-09-26 00:45:02+0000', tz='UTC'), Timestamp('2026-09-26 01:00:01+0000', tz='UTC'), Timestamp('2026-09-26 01:15:02+0000', tz='UT

In [89]:
one = df[df["station_code"] == df["station_code"].iloc[0]]
print(one.loc[one["capacity"].isna(), "collected_at_utc"].to_list())

[Timestamp('2026-09-02 22:45:02+0000', tz='UTC'), Timestamp('2026-09-02 23:00:02+0000', tz='UTC'), Timestamp('2026-09-02 23:15:01+0000', tz='UTC'), Timestamp('2026-09-02 23:30:02+0000', tz='UTC'), Timestamp('2026-09-02 23:45:02+0000', tz='UTC'), Timestamp('2026-09-03 00:00:01+0000', tz='UTC'), Timestamp('2026-09-03 00:15:02+0000', tz='UTC'), Timestamp('2026-09-03 00:45:01+0000', tz='UTC'), Timestamp('2026-09-25 22:30:01+0000', tz='UTC'), Timestamp('2026-09-25 22:45:02+0000', tz='UTC'), Timestamp('2026-09-25 23:15:02+0000', tz='UTC'), Timestamp('2026-09-25 23:30:02+0000', tz='UTC'), Timestamp('2026-09-25 23:45:01+0000', tz='UTC'), Timestamp('2026-09-26 00:00:02+0000', tz='UTC'), Timestamp('2026-09-26 00:15:01+0000', tz='UTC'), Timestamp('2026-09-26 00:30:02+0000', tz='UTC'), Timestamp('2026-09-26 00:45:02+0000', tz='UTC'), Timestamp('2026-09-26 01:00:01+0000', tz='UTC'), Timestamp('2026-09-26 01:15:02+0000', tz='UTC'), Timestamp('2026-09-26 01:30:01+0000', tz='UTC'), Timestamp('2026-09-

In [90]:
df["capacity"] = df.groupby("station_code")["capacity"].transform("first")
assert df["capacity"].isna().sum() == 0

In [94]:
print("\nbikes ≠ mech+elec:",
      (df["bikes_available"] != df["mechanical_bikes"] + df["electric_bikes"]).sum())
print("bikes+docks > capacity:",
      (df["bikes_available"] + df["docks_available"] > df["capacity"]).sum())
print("bikes+docks < capacity:",
      (df["bikes_available"] + df["docks_available"] < df["capacity"]).sum())



bikes ≠ mech+elec: 0
bikes+docks > capacity: 2256
bikes+docks < capacity: 169341


In [95]:
lab = lab.dropna(subset=["y"]).copy()
lab["y"] = lab["y"].astype(int)
print("labelled rows:", len(lab))
print("positive rate:", lab["y"].mean())

labelled rows: 327290
positive rate: 0.11187937303308992


In [75]:
print("rows:", len(df))                                       # 329 680
print("stations:", df["station_code"].nunique())              # 80
print("range:", df["collected_at_utc"].min(), "→", df["collected_at_utc"].max())
print("snapshots:", df["collected_at_utc"].nunique())         # 4 121
print("rows per station:", len(df) / df["station_code"].nunique())  # 4 121.0

rows: 329680
stations: 80
range: 2026-08-25 11:57:26+00:00 → 2026-10-08 20:15:01+00:00
snapshots: 4139
rows per station: 4121.0


In [78]:
print(df.groupby("station_code")["capacity"].nunique().value_counts())

capacity
1    80
Name: count, dtype: int64


In [79]:
miss = df[df["capacity"].isna()]
print(miss["collected_at_utc"].agg(["min", "max"]))

min   2026-09-02 22:30:01+00:00
max   2026-09-26 03:00:01+00:00
Name: collected_at_utc, dtype: datetime64[us, UTC]


In [80]:
print("range:", df["collected_at_utc"].min(), "->", df["collected_at_utc"].max())
print("snapshots:", df["collected_at_utc"].nunique())

range: 2026-08-25 11:57:26+00:00 -> 2026-10-08 20:15:01+00:00
snapshots: 4139


In [55]:
# Stations per distinct timestamp
per_ts = df.groupby("collected_at_utc")["station_code"].nunique()
print(per_ts.value_counts().sort_index())      # how many timestamps have 80, 79, ... stations?
print(per_ts[per_ts < 80].head(20))            # the incomplete ones

# After rounding to 15 minutes
df["slot"] = df["collected_at_utc"].dt.round("15min")
print("slots:", df["slot"].nunique())
per_slot = df.groupby("slot")["station_code"].nunique()
print(per_slot.value_counts().sort_index())
print("duplicates (station, slot):", df.duplicated(["station_code", "slot"]).sum())

station_code
30      18
50      18
80    4103
Name: count, dtype: int64
collected_at_utc
2026-08-28 04:00:02+00:00    30
2026-08-28 05:00:02+00:00    50
2026-08-28 05:30:02+00:00    30
2026-08-28 05:45:01+00:00    50
2026-09-02 10:45:01+00:00    30
2026-09-02 11:00:02+00:00    50
2026-09-02 11:15:01+00:00    50
2026-09-02 12:00:01+00:00    30
2026-09-02 13:15:01+00:00    50
2026-09-02 13:30:02+00:00    30
2026-09-08 03:30:01+00:00    30
2026-09-08 03:45:02+00:00    50
2026-09-08 07:30:01+00:00    30
2026-09-08 08:30:02+00:00    50
2026-09-08 08:45:01+00:00    30
2026-09-08 09:00:02+00:00    50
2026-09-08 09:30:02+00:00    30
2026-09-08 09:45:02+00:00    50
2026-09-14 02:45:02+00:00    50
2026-09-14 03:15:01+00:00    30
Name: station_code, dtype: int64
slots: 4138
station_code
30      18
50      18
80    4102
Name: count, dtype: int64
duplicates (station, slot): 80


In [81]:
df = df.sort_values(["station_code", "collected_at_utc"]).reset_index(drop=True)

In [82]:
# 1.Coverage
print("station number:", df["station_code"].nunique())
print("snapshots number :", df["collected_at_utc"].nunique())
print("time range:", df["collected_at_utc"].min(), "→", df["collected_at_utc"].max())
print("station info raws count:\n", df.groupby("station_code").size().describe())

station number: 80
snapshots number : 4139
time range: 2026-08-25 11:57:26+00:00 → 2026-10-08 20:15:01+00:00
station info raws count:
 count      80.0
mean     4121.0
std         0.0
min      4121.0
25%      4121.0
50%      4121.0
75%      4121.0
max      4121.0
dtype: float64
